# Real-Time Web Search with Llama Using free-web-search-ultimate

This notebook demonstrates how to give Llama models real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no additional API keys required**.

[![free-web-search-ultimate](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a tool for Llama models
- How to build a research assistant with real-time web access
- How to handle tool use with Llama's function calling API
- How to combine web search with Llama's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool (`free-web-search`) |

In [ ]:
# Install required packages
%pip install free-web-search-ultimate groq --quiet
# Note: This example uses Groq for fast Llama inference, but you can use any Llama API

In [ ]:
import json
import os
from groq import Groq
from free_web_search.search_web import UltimateSearcher

# Initialize the Groq client (for Llama inference)
client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

# Initialize the searcher
searcher = UltimateSearcher()


def web_search(
    query: str,
    max_results: int = 5,
    timelimit: str | None = None,
) -> str:
    """Search the web for current information.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return. Defaults to 5.
        timelimit: Optional time filter ('d' for day, 'w' for week,
            'm' for month, 'y' for year).

    Returns:
        A formatted string containing search results with titles, URLs,
        and snippets.
    """
    results = searcher.search(query, max_results=max_results, timelimit=timelimit)
    if not results:
        return "No results found."
    output = []
    for r in results:
        output.append(f"Title: {r.get('title', 'N/A')}")
        output.append(f"URL: {r.get('url', 'N/A')}")
        output.append(f"Snippet: {r.get('body', 'N/A')}")
        output.append("")
    return "\n".join(output)


def news_search(
    query: str,
    max_results: int = 5,
    timelimit: str | None = "w",
) -> str:
    """Search for recent news articles.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return. Defaults to 5.
        timelimit: Time filter for news recency ('d' for day, 'w' for week,
            'm' for month). Defaults to 'w' (past week).

    Returns:
        A formatted string containing news results with titles, URLs,
        and snippets.
    """
    results = searcher.search_news(query, max_results=max_results, timelimit=timelimit)
    if not results:
        return "No news found."
    output = []
    for r in results:
        output.append(f"Title: {r.get('title', 'N/A')}")
        output.append(f"URL: {r.get('url', 'N/A')}")
        output.append(f"Snippet: {r.get('body', 'N/A')}")
        output.append("")
    return "\n".join(output)


## Build the Research Assistant

In [ ]:
def research_with_llama(
    question: str,
    model: str = "llama-3.3-70b-versatile",
    max_iterations: int = 10,
) -> str:
    """Run a research query using Llama (via Groq) with web search tools.

    Args:
        question: The research question to answer.
        model: The Llama model to use via Groq. Defaults to
            'llama-3.3-70b-versatile'.
        max_iterations: Maximum number of tool-call iterations to prevent
            infinite loops. Defaults to 10.

    Returns:
        The final answer string from the model.
    """
    tools = [
        {
            "type": "function",
            "function": {
                "name": "web_search",
                "description": (
                    "Search the web for current information. Use for general queries "
                    "and topics that may have changed recently."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "The search query"},
                        "max_results": {
                            "type": "integer",
                            "description": "Maximum number of results (default: 5)",
                        },
                        "timelimit": {
                            "type": "string",
                            "description": "Time filter: 'd' (day), 'w' (week), 'm' (month), 'y' (year)",
                        },
                    },
                    "required": ["query"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "news_search",
                "description": (
                    "Search for recent news articles. Use for current events "
                    "and time-sensitive topics."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "The news search query"},
                        "max_results": {
                            "type": "integer",
                            "description": "Maximum number of results (default: 5)",
                        },
                        "timelimit": {
                            "type": "string",
                            "description": "Time filter: 'd' (day), 'w' (week), 'm' (month)",
                        },
                    },
                    "required": ["query"],
                },
            },
        },
    ]

    messages = [{"role": "user", "content": question}]

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                if fn_name == "web_search":
                    result = web_search(**fn_args)
                elif fn_name == "news_search":
                    result = news_search(**fn_args)
                else:
                    result = f"Unknown tool: {fn_name}"
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": result,
                })
        else:
            return msg.content

    return f"Reached maximum iterations ({max_iterations}) without a final answer."


## Example: Research with Llama

In [ ]:
# Set to True to run the live search examples below.
# Leave as False to avoid outbound network requests on fresh checkout.
RUN_EXAMPLES = False


In [ ]:
if RUN_EXAMPLES:
    # Example: Ask about current AI developments
    answer = research_with_llama(
        "What are the latest developments in open-source LLMs this week?"
    )
    print("\n--- Final Answer ---")
    print(answer)

## Direct Usage

You can also use the package directly without any LLM.

In [ ]:
from free_web_search.search_web import UltimateSearcher
import json

# Initialize the searcher
_searcher = UltimateSearcher()

def search(query: str, max_results: int = 5) -> list:
    """Search the web for real-time information."""
    answer = _searcher.search(query)
    return [
        {'title': s.title, 'url': s.url, 'snippet': s.snippet}
        for s in answer.sources[:max_results]
    ]

def search_news(query: str, max_results: int = 5) -> list:
    """Search for recent news articles."""
    answer = _searcher.search(query, search_type='news')
    return [
        {'title': s.title, 'url': s.url, 'snippet': s.snippet, 'date': getattr(s, 'date', '')}
        for s in answer.sources[:max_results]
    ]

# Direct web search
results = search("Meta Llama latest updates", max_results=3)
for r in results:
    print(f"Title: {r.get('title', 'N/A')}")
    print(f"URL: {r.get('url', r.get('href', 'N/A'))}")
    print()

## MCP Server Usage

The `free-web-search-ultimate` package also works as an MCP server:

```bash
# Install and run as MCP server
pip install free-web-search-ultimate
free-web-search-mcp

# Or use as CLI
search-web "latest Llama news"
```

## Summary

In this notebook, we demonstrated how to:

1. **Install** `free-web-search-ultimate` — a zero-cost, privacy-first web search package
2. **Define function tools** for Llama models using the OpenAI-compatible API
3. **Build a research assistant** with an agentic tool-use loop
4. **Research complex topics** by combining Llama's reasoning with real-time web search

### Resources

- [free-web-search-ultimate on GitHub](https://github.com/wd041216-bit/free-web-search-ultimate)
- [free-web-search-ultimate on PyPI](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [Llama Function Calling Documentation](https://llama.meta.com/docs/model-cards-and-prompt-formats/llama3_1/)